# Compute engines: numba, PyTorch, and symbolic

mixle separates what a model is (distributions own their likelihoods and sufficient statistics) from how it is evaluated (a compute engine supplies the array arithmetic). One model definition therefore runs unchanged on several backends:

- the numpy engine - and, for models built from declared exponential-family leaves, a numba kernel is now compiled and selected automatically;
- the PyTorch engine - batched scoring on CPU/GPU/MPS, plus gradient-based fitting;
- the symbolic engine - which traces a model's math into an expression tree you can export to a computer-algebra system.

This notebook fits the same mixture three ways (and verifies they agree), then uses the symbolic engine to read a distribution's log-density as algebra and differentiate it.

These features live in the current mixle release.


In [1]:
import io, time
import numpy as np
from mixle.stats import (GaussianDistribution, PoissonDistribution, OptionalDistribution,
                        CompositeDistribution, MixtureDistribution, GaussianEstimator,
                        PoissonEstimator, OptionalEstimator, CompositeEstimator,
                        MixtureEstimator, seq_encode, seq_log_density_sum)
from mixle.engines import NUMPY_ENGINE, TorchEngine, SYMBOLIC_ENGINE, to_sympy, to_sage, to_latex
from mixle.inference.estimation import optimize

# a heterogeneous mixture: each component is a record of a real value, a count,
# and a sometimes-missing count
d1 = CompositeDistribution((GaussianDistribution(-2.0, 1.0), PoissonDistribution(3.0),
                            OptionalDistribution(PoissonDistribution(4.0), p=0.2)))
d2 = CompositeDistribution((GaussianDistribution(2.0, 1.0), PoissonDistribution(8.0),
                            OptionalDistribution(PoissonDistribution(9.0), p=0.05)))
true = MixtureDistribution([d1, d2], [0.5, 0.5])
data = true.sampler(seed=1).sample(40000)

est = MixtureEstimator([CompositeEstimator((GaussianEstimator(), PoissonEstimator(),
        OptionalEstimator(PoissonEstimator(), est_prob=True)))] * 2)
print('n =', len(data))

n = 40000


## The same EM, three engines

`optimize` runs EM. With no engine it uses the classic accumulator path. Pass an engine and the per-iteration E-step runs through a compiled kernel instead - and on the numpy engine, a model built from declared exponential-family leaves automatically gets a numba kernel. The M-step is the same estimator code in every case, so the fits are identical; only the scoring/accumulation is swapped.


In [2]:
def run(engine, label):
    t0 = time.perf_counter()
    m = optimize(data, est, max_its=15, rng=np.random.RandomState(2), out=io.StringIO(), engine=engine)
    dt = time.perf_counter() - t0
    _, ll = seq_log_density_sum(seq_encode(data, model=m), m)
    print('%-22s ll=%.4f   15 EM iterations in %.2fs' % (label, ll, dt))
    return m

m_legacy = run(None,          'legacy accumulator')
m_numba  = run(NUMPY_ENGINE,  'numpy engine (numba)')
m_torch  = run(TorchEngine(), 'torch engine')

legacy accumulator     ll=-265963.0622   15 EM iterations in 0.08s


numpy engine (numba)   ll=-265963.0622   15 EM iterations in 0.63s


torch engine           ll=-265963.0622   15 EM iterations in 0.23s


All three reach the same log-likelihood. Which kernel did the numpy engine actually choose? Ask the model:


In [3]:
kernel = true.kernel()                      # default engine is numpy
print('default numpy kernel :', type(kernel).__name__)
print('torch engine kernel  :', type(true.kernel(engine=TorchEngine())).__name__)

# the numba kernel scores identically to the distribution's own reference path
enc1 = true.dist_to_encoder().seq_encode(data[:2000])
ok = np.allclose(kernel.score(enc1), true.seq_log_density(enc1), rtol=1e-10, atol=1e-10)
print('numba kernel matches the reference seq path to 1e-10:', ok)

default numpy kernel : GeneratedNumbaKernel
torch engine kernel  : StackedMixtureKernel
numba kernel matches the reference seq path to 1e-10: True


`GeneratedNumbaKernel` is built from each leaf's exponential-family declaration - the library lowers `log h(x) + T(x)·η(θ) − A(η)` to a compiled scalar loop. Nothing in your model definition changes; acceleration is a property of the engine, not the model.

Not every model is numba-eligible: some leaves inside a composite (a `CategoricalDistribution`, today) have no stacked exponential-family declaration yet, so the model falls back to the generic kernel - still correct, just not compiled. You never have to think about this: `model.kernel()` always returns a working kernel, the fastest one available.


In [4]:
from mixle.stats import CategoricalDistribution
plain = MixtureDistribution([GaussianDistribution(-2, 1), GaussianDistribution(2, 1)], [0.5, 0.5])
with_cat = MixtureDistribution([CompositeDistribution((GaussianDistribution(-2, 1), CategoricalDistribution({'a': 0.7, 'b': 0.3}))),
                                CompositeDistribution((GaussianDistribution(2, 1), CategoricalDistribution({'a': 0.3, 'b': 0.7})))], [0.5, 0.5])
print('plain Gaussian mixture     ->', type(plain.kernel()).__name__)
print('mixture with a categorical ->', type(with_cat.kernel()).__name__, '(falls back, still correct)')

plain Gaussian mixture     -> GeneratedNumbaKernel
mixture with a categorical -> GenericKernel (falls back, still correct)


The PyTorch engine instead scores the whole mixture as batched tensors, which is what enables GPU execution and autograd-based fitting (`fit_mle` / `fit_map` - see the gradient and MAP fitting notebook).


## The symbolic engine: a model's math as algebra

Because every distribution writes its log-density through the engine's arithmetic, running a distribution under the symbolic engine doesn't compute a number - it records the expression. That expression exports to SymPy (https://www.sympy.org) - or SageMath (https://www.sagemath.org) via `to_sage` - for simplification, LaTeX, or calculus. Here is a Gaussian's log-density, traced and then exported:


In [5]:
from IPython.display import Math, display
x = SYMBOLIC_ENGINE.symbol('x')
expr = GaussianDistribution(0.5, 2.0).backend_seq_log_density(x, SYMBOLIC_ENGINE)
sym = to_sympy(expr)                       # the traced log-density as a SymPy expression
display(Math(r'\log p(x) = ' + to_latex(sym)))

<IPython.core.display.Math object>

In [6]:
import sympy
xs = next(iter(sym.free_symbols))
score = sympy.simplify(sympy.diff(sym, xs))         # symbolic differentiation
display(Math(r'\frac{d}{dx}\log p(x) = ' + sympy.latex(score)))
analytic = sympy.simplify(-(xs - 0.5) / 2.0)        # -(x - mu)/sigma^2,  mu=0.5, sigma^2=2
print('matches the analytic score -(x-mu)/sigma^2 :', sympy.simplify(score - analytic) == 0)

<IPython.core.display.Math object>

matches the analytic score -(x-mu)/sigma^2 : True


The symbolic engine traced the Gaussian's log-density, SymPy differentiated it, and the result is exactly the analytic score $-(x-\mu)/\sigma^2$ - the same quantity the gradient fitter computes numerically. This is useful for inspecting generated kernels, checking derivations, or rendering models to LaTeX:


In [7]:
# to_latex returns the LaTeX string itself, for dropping a model's math into a paper:
unit = GaussianDistribution(0.0, 1.0).backend_seq_log_density(SYMBOLIC_ENGINE.symbol('x'), SYMBOLIC_ENGINE)
print(to_latex(to_sympy(unit)))

- 0.5 x^{2} - 0.918938533204673


In [8]:
# The same traced expression exports to SageMath through to_sage, mirroring to_sympy.
# It needs Sage installed and run under its interpreter; here we show the call and
# its honest result in a plain-Python environment.
try:
    print('sage:', to_sage(expr))
except ImportError as err:
    print('to_sage ->', err)

sage: -0.5*(x - 0.5)*(0.5*x - 0.25) - 1.2655121234846454


## Fitting on the torch engine

Passing `engine=TorchEngine()` to `optimize` runs the same EM with the E-step/scoring on the torch ComputeEngine (GPU-capable where a device is available). It is verified to match the NumPy result - the model definition never changes, only where the arithmetic runs.


In [9]:
from mixle.stats import (GaussianDistribution, PoissonDistribution, CompositeDistribution,
                        GaussianEstimator, PoissonEstimator, CompositeEstimator,
                        seq_log_density_sum, seq_encode)
import numpy as np, io

# a Composite leaf (real value + count); EM here has a unique closed-form optimum,
# so the two engines must land on identical parameters.
truth = CompositeDistribution((GaussianDistribution(2.0, 1.5), PoissonDistribution(4.0)))
cdata = truth.sampler(seed=0).sample(5000)
cest = CompositeEstimator((GaussianEstimator(), PoissonEstimator()))
ll = lambda m: seq_log_density_sum(seq_encode(cdata, model=m), m)[1]

m_np = optimize(cdata, cest, max_its=1, rng=np.random.RandomState(1), out=io.StringIO())
m_t  = optimize(cdata, cest, max_its=1, rng=np.random.RandomState(1), engine=TorchEngine(), out=io.StringIO())
fmt = lambda m: '(mu=%.4f, sigma2=%.4f, lam=%.4f)' % (m.dists[0].mu, m.dists[0].sigma2, m.dists[1].lam)
print('numpy :', fmt(m_np), ' LL %.3f' % ll(m_np))
print('torch :', fmt(m_t),  ' LL %.3f' % ll(m_t))
print('identical:', np.isclose(ll(m_np), ll(m_t)))

numpy : (mu=2.0112, sigma2=1.5133, lam=4.0040)  LL -18614.150
torch : (mu=2.0112, sigma2=1.5133, lam=4.0040)  LL -18614.150
identical: True


## Engine introspection and `torch.compile`

Two practical controls. First, capabilities: each family declares which engines it supports and what kind of kernel it uses, queryable with `capabilities_for`. The planner uses this to choose an engine and to fall back to numpy for families that are intentionally numpy-only (e.g. permutation-distance models). Second, compilation: `TorchEngine(compile=True)` routes the scoring kernel through `torch.compile`, fusing the elementwise exponential-family arithmetic into a single fused kernel - an opt-in speedup for the torch path (off by default to avoid the one-time warmup cost). The model definition and the scoring call are unchanged either way.


In [10]:
from mixle.stats import capabilities_for, MallowsDistribution
from mixle.stats.compute.backend import backend_seq_log_density

# which engines does each family support, and what kernel backs it?
for d in [GaussianDistribution(0.0, 1.0), PoissonDistribution(4.0), MallowsDistribution([0, 1, 2], 1.0)]:
    cap = capabilities_for(d)
    print('%-22s engines=%-18s kernel=%s' % (type(d).__name__, str(cap.engine_ready), cap.kernel_status))
    if cap.numpy_only_reason:
        print('%-22s   (numpy-only: %s)' % ('', cap.numpy_only_reason))

# torch.compile is a per-engine flag; nothing else about the call changes
gd = GaussianDistribution(0.0, 1.0)
enc = gd.dist_to_encoder().seq_encode([-1.0, 0.0, 1.0, 2.0])
eng = TorchEngine(compile=True)
print('\ncompile_enabled :', eng.compile_enabled)
print('torch scores    :', np.round(np.asarray(eng.to_numpy(backend_seq_log_density(gd, enc, eng))), 4))
print('numpy scores    :', np.round(np.asarray(gd.seq_log_density(enc)), 4), '(identical)')

GaussianDistribution   engines=('numpy', 'torch') kernel=numba_adapter
PoissonDistribution    engines=('numpy', 'torch') kernel=numba_adapter
MallowsDistribution    engines=('numpy',)         kernel=numpy_only
                         (numpy-only: Kendall tau distance over permutation pairs is numpy-native.)

compile_enabled : True
torch scores    : [-1.4189 -0.9189 -1.4189 -2.9189]
numpy scores    : [-1.4189 -0.9189 -1.4189 -2.9189] (identical)


## When to use which engine

| engine | reach for it when |
|---|---|
| numpy (default) | everyday fitting; exponential-family models get a numba kernel automatically |
| PyTorch | GPU/MPS execution, gradient `fit_mle` / `fit_map`, very large batched scoring |
| symbolic | inspecting generated kernels, exporting model math to SymPy/SageMath, symbolic calculus and LaTeX |

The engine is chosen at the call site (`optimize(..., engine=...)` or `model.kernel(engine=...)`); the model definition never changes. Distributed and streaming estimation compose with all of this - see the distributed and streaming estimation notebook.
